# Implementación secuencial de K-Means en Go

En esta sección se desarrolla una implementación secuencial del algoritmo K-Means utilizando Go. El objetivo es agrupar mediciones de calidad del aire según sus concentraciones de PM10 y PM2.5.

Para ello, se utiliza un conjunto de datos previamente procesado que contiene 445,347 vectores, donde cada vector representa una observación con dos características: PM10 y PM2.5.

La implementación se desarrolla de manera secuencial y servirá como referencia para comparar posteriormente su rendimiento con una versión paralela del algoritmo.


## 1. Configuración del entorno

Primero se instala Go en Google Colab y se verifica que esté correctamente configurado.

También se revisan las características del entorno de ejecución, como el procesador, el sistema operativo y la memoria disponible. Esta información será útil posteriormente para analizar los tiempos de ejecución del algoritmo.

In [1]:
!apt-get update
!apt-get install golang-go

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,119 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,762 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.5 MB]
Get:14 http://arc

In [2]:
!go version
!uname -sr
!lscpu
!free -h

go version go1.22.2 linux/amd64
Linux 6.6.159+
Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   79
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                0
    BogoMIPS:                4400.30
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma c

## 2. Acceso al conjunto de datos

Se conecta Google Drive con Google Colab para acceder al archivo CSV que contiene las mediciones de calidad del aire.

El archivo utilizado corresponde a las observaciones de una hora y contiene las columnas PM10 y PM2.5, que serán las características utilizadas por el algoritmo K-Means.

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 3. Definición de la estructura del conjunto de datos — `dataset.go`

En este archivo se define la estructura `Dataset`, que permite almacenar los datos que utilizará el algoritmo.

Se incluyen tres atributos:

- `Points`: almacena los vectores de datos. Cada vector contiene los valores de PM10 y PM2.5.
- `N`: representa la cantidad total de vectores.
- `D`: representa la cantidad de dimensiones de cada vector. En este caso, su valor es 2.

También se implementa `NewDataset`, una función que recibe los puntos y construye un objeto `Dataset` con la información correspondiente.

In [4]:
%%writefile dataset.go
package main

// Dimensions representa la cantidad de características
// que utilizaremos para cada observación: PM10 y PM2.5.
const Dimensions = 2

// Dataset almacena los vectores que utilizaremos en K-Means.
type Dataset struct {
	Points [][]float64
	N      int
	D      int
}

// NewDataset crea un conjunto de datos a partir de una lista de puntos.
func NewDataset(points [][]float64) Dataset {
	return Dataset{
		Points: points,
		N:      len(points),
		D:      Dimensions,
	}
}


Writing dataset.go


## 4. Carga de datos — `loader.go`

En este archivo se implementa la función `LoadCSV`, encargada de leer el conjunto de datos desde un archivo CSV.

Primero, se abre el archivo y se verifica que contenga las columnas PM10 y PM2.5. Luego, se recorren sus registros y se convierten los valores de texto a números decimales (`float64`).

Cada registro se almacena como un vector de dos dimensiones: `[PM10, PM2.5]`.

Finalmente, los vectores se utilizan para construir un objeto `Dataset`, que será entregado al algoritmo K-Means. También se contemplan errores relacionados con la apertura y lectura del archivo, así como con la conversión de los valores.

In [5]:
%%writefile loader.go
package main

import (
	"encoding/csv"
	"fmt"
	"io"
	"math"
	"os"
	"strconv"
)

// LoadCSV lee un archivo CSV con las columnas PM10 y PM2.5.
func LoadCSV(path string) (Dataset, error) {
	file, err := os.Open(path)
	if err != nil {
		return Dataset{}, fmt.Errorf("no se pudo abrir el archivo: %w", err)
	}
	defer file.Close()

	reader := csv.NewReader(file)

	header, err := reader.Read()
	if err != nil {
		return Dataset{}, fmt.Errorf("no se pudo leer el encabezado: %w", err)
	}

	if len(header) != 2 || header[0] != "PM10" || header[1] != "PM2.5" {
		return Dataset{}, fmt.Errorf("formato de CSV incorrecto: se esperaban PM10 y PM2.5")
	}

	points := make([][]float64, 0)

	for row := 2; ; row++ {
		record, err := reader.Read()

		if err == io.EOF {
			break
		}

		if err != nil {
			return Dataset{}, fmt.Errorf("error al leer la fila %d: %w", row, err)
		}

		if len(record) != 2 {
			return Dataset{}, fmt.Errorf("la fila %d debe tener exactamente 2 columnas", row)
		}

		pm10, err := strconv.ParseFloat(record[0], 64)
		if err != nil {
			return Dataset{}, fmt.Errorf("valor PM10 inválido en la fila %d: %w", row, err)
		}

		pm25, err := strconv.ParseFloat(record[1], 64)
		if err != nil {
			return Dataset{}, fmt.Errorf("valor PM2.5 inválido en la fila %d: %w", row, err)
		}

		if math.IsNaN(pm10) || math.IsInf(pm10, 0) ||
			math.IsNaN(pm25) || math.IsInf(pm25, 0) {
			return Dataset{}, fmt.Errorf("se encontró un valor no finito en la fila %d", row)
		}

		points = append(points, []float64{pm10, pm25})
	}

	if len(points) == 0 {
		return Dataset{}, fmt.Errorf("el CSV no contiene datos")
	}

	return NewDataset(points), nil
}


Writing loader.go


## 5. Implementación del algoritmo K-Means — `kmeans.go`

En este archivo se desarrolla la lógica principal del algoritmo K-Means. Su objetivo es agrupar los vectores según la distancia entre sus valores de PM10 y PM2.5, buscando que los puntos de un mismo clúster tengan características similares.

La implementación se divide en las siguientes operaciones:

**1. Inicialización de centroides**

Se seleccionan aleatoriamente K puntos del conjunto de datos como centroides iniciales. Se utiliza una semilla fija (`seed = 42`) para que la selección sea reproducible entre ejecuciones.

**2. Asignación de clústeres**

Cada punto se compara con los centroides utilizando la distancia euclidiana al cuadrado. Luego, se asigna al clúster cuyo centroide se encuentre más cerca.

**3. Actualización de centroides**

Se calcula el promedio de los valores de PM10 y PM2.5 de los puntos pertenecientes a cada clúster. Estos promedios se utilizan como nuevos centroides.

Si un clúster queda vacío, se conserva su centroide anterior.

**4. Criterio de convergencia**

Después de actualizar los centroides, se calcula cuánto se han desplazado respecto a sus posiciones anteriores. Si el desplazamiento total es menor que la tolerancia establecida (`0.001`), el algoritmo se considera convergido.

El proceso se repite hasta alcanzar la convergencia o llegar al máximo de iteraciones permitido (`100`).

**5. Cálculo de la inercia**

Al finalizar, se calcula la inercia como la suma de las distancias euclidianas al cuadrado entre cada punto y el centroide de su clúster. Este valor permite evaluar qué tan compactos son los grupos obtenidos.

Finalmente, se devuelve una estructura `KMeansResult` que contiene los centroides finales, las etiquetas asignadas, la inercia, el número de iteraciones y el estado de convergencia.

In [6]:

%%writefile kmeans.go
package main

import (
	"fmt"
	"math"
	"math/rand"
)

// initializeCentroids selecciona K puntos aleatorios del dataset
// para utilizarlos como centroides iniciales.
func initializeCentroids(data Dataset, k int, seed int64) ([][]float64, error) {
	if err := validateDataset(data); err != nil {
		return nil, err
	}

	if k <= 0 || k > data.N {
		return nil, fmt.Errorf("K debe estar entre 1 y N")
	}

	rng := rand.New(rand.NewSource(seed))
	indices := rng.Perm(data.N)[:k]

	centroids := make([][]float64, k)

	for i, index := range indices {
		centroids[i] = make([]float64, data.D)
		copy(centroids[i], data.Points[index])
	}

	return centroids, nil
}

// validateDataset comprueba que el conjunto de datos tenga
// una estructura válida para trabajar con K-Means.
func validateDataset(data Dataset) error {
	if data.N <= 0 || len(data.Points) == 0 {
		return fmt.Errorf("el conjunto de datos está vacío")
	}

	if data.N != len(data.Points) {
		return fmt.Errorf("la cantidad de puntos no coincide con N")
	}

	if data.D != 2 {
		return fmt.Errorf("el dataset debe tener exactamente 2 dimensiones")
	}

	for i, point := range data.Points {
		if len(point) != data.D {
			return fmt.Errorf("el punto %d tiene dimensiones inconsistentes", i)
		}

		for _, value := range point {
			if math.IsNaN(value) || math.IsInf(value, 0) {
				return fmt.Errorf("el punto %d contiene un valor no finito", i)
			}
		}
	}

	return nil
}

// squaredDistance calcula la distancia euclidiana al cuadrado
// entre dos puntos.
func squaredDistance(a, b []float64) float64 {
	var distance float64

	for i := range a {
		diff := a[i] - b[i]
		distance += diff * diff
	}

	return distance
}

// assignClusters asigna cada punto al clúster cuyo centroide
// se encuentra más cerca.
func assignClusters(data Dataset, centroids [][]float64) []int {
	labels := make([]int, data.N)

	for i, point := range data.Points {
		bestCluster := 0
		minDistance := squaredDistance(point, centroids[0])

		for j := 1; j < len(centroids); j++ {
			distance := squaredDistance(point, centroids[j])

			if distance < minDistance {
				minDistance = distance
				bestCluster = j
			}
		}

		labels[i] = bestCluster
	}

	return labels
}

// updateCentroids actualiza los centroides calculando el promedio
// de los puntos asignados a cada clúster.
// Si un clúster queda vacío, conserva su centroide anterior.
func updateCentroids(data Dataset, labels []int, centroids [][]float64) [][]float64 {
	k := len(centroids)

	newCentroids := make([][]float64, k)
	counts := make([]int, k)

	for i := range newCentroids {
		newCentroids[i] = make([]float64, data.D)
	}

	// Acumulamos los valores de cada dimensión por clúster.
	for i, point := range data.Points {
		cluster := labels[i]
		counts[cluster]++

		for d := 0; d < data.D; d++ {
			newCentroids[cluster][d] += point[d]
		}
	}

	// Calculamos el promedio de cada clúster.
	for cluster := 0; cluster < k; cluster++ {
		if counts[cluster] > 0 {
			for d := 0; d < data.D; d++ {
				newCentroids[cluster][d] /= float64(counts[cluster])
			}
		} else {
			copy(newCentroids[cluster], centroids[cluster])
		}
	}

	return newCentroids
}

// KMeansResult almacena los resultados obtenidos al ejecutar K-Means.
type KMeansResult struct {
	Centroids  [][]float64
	Labels     []int
	Inertia    float64
	Iterations int
	Converged  bool
}

// calculateInertia calcula la suma de las distancias al cuadrado
// entre cada punto y el centroide de su clúster.
func calculateInertia(data Dataset, centroids [][]float64, labels []int) float64 {
	var inertia float64

	for i, point := range data.Points {
		inertia += squaredDistance(point, centroids[labels[i]])
	}

	return inertia
}

// RunKMeans ejecuta el algoritmo K-Means secuencial.
func RunKMeans(data Dataset, k int, maxIter int, tol float64, seed int64) (KMeansResult, error) {
	if err := validateDataset(data); err != nil {
		return KMeansResult{}, err
	}

	if k <= 0 || k > data.N {
		return KMeansResult{}, fmt.Errorf("K debe estar entre 1 y N")
	}

	if maxIter <= 0 {
		return KMeansResult{}, fmt.Errorf("el máximo de iteraciones debe ser positivo")
	}

	if tol < 0 || math.IsNaN(tol) || math.IsInf(tol, 0) {
		return KMeansResult{}, fmt.Errorf("la tolerancia debe ser finita y no negativa")
	}

	centroids, err := initializeCentroids(data, k, seed)
	if err != nil {
		return KMeansResult{}, err
	}

	result := KMeansResult{}

	for iteration := 0; iteration < maxIter; iteration++ {
		labels := assignClusters(data, centroids)
		newCentroids := updateCentroids(data, labels, centroids)

		var movement float64

		for cluster := 0; cluster < k; cluster++ {
			movement += squaredDistance(centroids[cluster], newCentroids[cluster])
		}

		centroids = newCentroids
		result.Iterations = iteration + 1

		if math.Sqrt(movement) < tol {
			result.Converged = true
			break
		}
	}

	// Reasignamos los puntos usando los centroides finales.
	result.Centroids = centroids
	result.Labels = assignClusters(data, centroids)
	result.Inertia = calculateInertia(data, centroids, result.Labels)

	return result, nil
}



Writing kmeans.go


In [7]:
!go mod init kmeans-secuencial

go: creating new go.mod: module kmeans-secuencial
go: to add module requirements and sums:
	go mod tidy


## 6. Ejecución del algoritmo — `main.go`

Este archivo coordina la ejecución de la implementación secuencial.

Primero, se carga el conjunto de datos utilizando `LoadCSV` y se registra el tiempo que toma esta operación.

Luego, se ejecuta K-Means con los siguientes parámetros:

- K = 4 clústeres.
- Máximo de iteraciones = 100.
- Tolerancia = 0.001.
- Semilla = 42.

Se mide el tiempo de ejecución del algoritmo por separado del tiempo de carga de los datos.

Finalmente, se muestran los resultados obtenidos: número de iteraciones, convergencia, inercia, centroides finales y cantidad de puntos asignados a cada clúster.

Adicionalmente, se calculan estadísticas descriptivas del conjunto de datos, como promedios, valores máximos y percentiles de PM10 y PM2.5. Esto permite tener una referencia sobre la distribución de las concentraciones utilizadas en el agrupamiento.

In [11]:
%%writefile main.go

package main

import (
	"fmt"
	"log"
	"sort"
	"time"
)

// Procesa un dataset y muestra sus estadísticas y resultados de K-Means.
func procesarDataset(nombre string, ruta string) {
	fmt.Println("\n========================================")
	fmt.Println("DATASET:", nombre)
	fmt.Println("========================================")

	fmt.Println("\nCargando datos...")
	startLoad := time.Now()

	dataset, err := LoadCSV(ruta)
	if err != nil {
		log.Fatal(err)
	}

	loadTime := time.Since(startLoad)

	fmt.Println("Vectores:", dataset.N)
	fmt.Println("Dimensiones:", dataset.D)

	// Estadísticas básicas de los datos.
	var sumaPM10, sumaPM25 float64
	var maxPM10, maxPM25 float64
	var cerosAmbos int

	for _, point := range dataset.Points {
		pm10 := point[0]
		pm25 := point[1]

		sumaPM10 += pm10
		sumaPM25 += pm25

		if pm10 > maxPM10 {
			maxPM10 = pm10
		}

		if pm25 > maxPM25 {
			maxPM25 = pm25
		}

		if pm10 == 0 && pm25 == 0 {
			cerosAmbos++
		}
	}

	fmt.Println("\n--- ESTADÍSTICAS DE LOS DATOS ---")
	fmt.Printf("Promedio PM10: %.4f\n", sumaPM10/float64(dataset.N))
	fmt.Printf("Promedio PM2.5: %.4f\n", sumaPM25/float64(dataset.N))
	fmt.Printf("Máximo PM10: %.4f\n", maxPM10)
	fmt.Printf("Máximo PM2.5: %.4f\n", maxPM25)
	fmt.Printf("Vectores con ambas concentraciones en cero: %d\n", cerosAmbos)

	// Preparamos los valores para calcular los percentiles.
	pm10Values := make([]float64, dataset.N)
	pm25Values := make([]float64, dataset.N)

	for i, point := range dataset.Points {
		pm10Values[i] = point[0]
		pm25Values[i] = point[1]
	}

	fmt.Println("\n--- PERCENTILES ---")
	fmt.Printf("PM10 P50: %.2f\n", calculatePercentile(pm10Values, 0.50))
	fmt.Printf("PM10 P90: %.2f\n", calculatePercentile(pm10Values, 0.90))
	fmt.Printf("PM10 P95: %.2f\n", calculatePercentile(pm10Values, 0.95))
	fmt.Printf("PM10 P99: %.2f\n", calculatePercentile(pm10Values, 0.99))

	fmt.Printf("PM2.5 P50: %.2f\n", calculatePercentile(pm25Values, 0.50))
	fmt.Printf("PM2.5 P90: %.2f\n", calculatePercentile(pm25Values, 0.90))
	fmt.Printf("PM2.5 P95: %.2f\n", calculatePercentile(pm25Values, 0.95))
	fmt.Printf("PM2.5 P99: %.2f\n", calculatePercentile(pm25Values, 0.99))

	// Ejecutamos K-Means secuencial.
	fmt.Println("\nEjecutando K-Means secuencial...")

	start := time.Now()

	result, err := RunKMeans(dataset, 4, 100, 0.001, 42)
	if err != nil {
		log.Fatal(err)
	}

	elapsed := time.Since(start)

	fmt.Println("\n--- RESULTADOS DE K-MEANS ---")
	fmt.Println("Iteraciones:", result.Iterations)
	fmt.Println("Convergencia:", result.Converged)
	fmt.Println("Inercia:", result.Inertia)
	fmt.Println("Tiempo de carga:", loadTime)
	fmt.Println("Tiempo de ejecución:", elapsed)

	fmt.Println("\nCentroides:")
	for i, centroid := range result.Centroids {
		fmt.Printf("Clúster %d: PM10=%.4f, PM2.5=%.4f\n",
			i, centroid[0], centroid[1])
	}

	counts := make([]int, 4)
	for _, label := range result.Labels {
		counts[label]++
	}

	fmt.Println("\nCantidad de puntos por clúster:")
	for i, count := range counts {
		fmt.Printf("Clúster %d: %d\n", i, count)
	}
}

// Calcula el percentil indicado sobre una copia de los valores.
func calculatePercentile(values []float64, percentile float64) float64 {
	sorted := make([]float64, len(values))
	copy(sorted, values)

	sort.Float64s(sorted)

	index := int(float64(len(sorted)-1) * percentile)
	return sorted[index]
}

func main() {
	ruta1Hora := "/content/drive/MyDrive/Concurrente/Vectores_PM/20261105T005132_156935Z/features_1_hora.csv"
	ruta24Horas := "/content/drive/MyDrive/Concurrente/Vectores_PM/20261105T005132_156935Z/features_24_horas.csv"

	procesarDataset("1 HORA", ruta1Hora)
	procesarDataset("24 HORAS", ruta24Horas)
}



Overwriting main.go


## 7. Ejecución y resultados

Una vez definidos todos los archivos, se inicializa el módulo de Go y se ejecuta el programa.

Durante la ejecución se carga el conjunto de datos y se aplica K-Means de forma secuencial. Se registran los resultados obtenidos y los tiempos de ejecución para utilizarlos posteriormente como referencia en la comparación con la implementación paralela.

La ejecución utiliza una configuración fija para mantener condiciones reproducibles durante las pruebas.

In [12]:
!go run .


DATASET: 1 HORA

Cargando datos...
Vectores: 445347
Dimensiones: 2

--- ESTADÍSTICAS DE LOS DATOS ---
Promedio PM10: 42.3842
Promedio PM2.5: 19.8080
Máximo PM10: 3848.7800
Máximo PM2.5: 1256.3000
Vectores con ambas concentraciones en cero: 16437

--- PERCENTILES ---
PM10 P50: 23.45
PM10 P90: 104.90
PM10 P95: 149.46
PM10 P99: 252.99
PM2.5 P50: 11.31
PM2.5 P90: 45.64
PM2.5 P95: 76.80
PM2.5 P99: 137.58

Ejecutando K-Means secuencial...

--- RESULTADOS DE K-MEANS ---
Iteraciones: 96
Convergencia: true
Inercia: 3.89845120224682e+08
Tiempo de carga: 2.005376706s
Tiempo de ejecución: 2.726374514s

Centroides:
Clúster 0: PM10=541.6889, PM2.5=117.2497
Clúster 1: PM10=18.4804, PM2.5=9.6445
Clúster 2: PM10=185.7345, PM2.5=92.2620
Clúster 3: PM10=80.5215, PM2.5=34.1009

Cantidad de puntos por clúster:
Clúster 0: 1333
Clúster 1: 330559
Clúster 2: 27648
Clúster 3: 85807

DATASET: 24 HORAS

Cargando datos...
Vectores: 18297
Dimensiones: 2

--- ESTADÍSTICAS DE LOS DATOS ---
Promedio PM10: 42.8914
Pro